# Displays data and creates datasets for testing

Daily data

In [ ]:
import importlib
import re
import pandas as pd
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import scipy
import plotly.graph_objects as go
import plotly.express as px
from performance import plotly_forecasting as go_d

## Load probabilistic forecasts and observations

In [ ]:
test_dataset_path = Path(r'tests\test_datasets_daily')

obs = pd.read_parquet(test_dataset_path / 'obs.parquet')
prob = pd.read_parquet(test_dataset_path / 'prob.parquet')

display(obs.head(5))
display(prob.head(5))

In [ ]:
leadtimes = [pd.Timedelta(f'{i}D') for i in [2, 5, 10]]

fig = go.Figure()
go_d.plot_lt_probabilistic(fig, prob, leadtimes=leadtimes, colors=px.colors.sequential.Plasma, color_loops=1)
go_d.add_observed_trace(fig, obs, showlegend=False)
go_d.apply_default_layout(fig, yaxis_title="Q [m3/s]")
fig.show()

In [ ]:
production_datetimes = prob.index.get_level_values('production_datetime').unique()[::15]

fig = go.Figure()
go_d.plot_pd_probabilistic(fig, prob, production_datetimes=production_datetimes, colors=px.colors.sequential.Plasma, color_loops=1)
go_d.add_observed_trace(fig, obs, showlegend=False)
go_d.apply_default_layout(fig, yaxis_title="Q [m3/s]")
fig.show()

## Handle deterministic forecasts
Created with the median of the probabilistic ones

In [ ]:
det_path = test_dataset_path / 'det.parquet'
if det_path.exists():
    det_parquet = pd.read_parquet(det_path)
else:
    det = (
        prob['Q']
        .unstack('non_exceedance')
        .reindex(columns=sorted([*prob.index.get_level_values('non_exceedance').unique(), 0.5]))
        .interpolate(axis=1)[0.5]
        .to_frame(name='Q')
    )
    det.to_parquet(det_path)

display(det.head(5))

In [ ]:
leadtimes = [pd.Timedelta(f'{i}D') for i in [2, 5, 10]]

fig = go.Figure()
go_d.plot_lt_deterministic(fig, det_parquet, leadtimes=leadtimes, colorscale=px.colors.sequential.Magma, showlegend=False)
go_d.add_observed_trace(fig, obs, showlegend=False)
go_d.apply_default_layout(fig, yaxis_title="Q [m3/s]")

In [ ]:
production_datetimes = det.index.get_level_values('production_datetime').unique()[::15]

fig = go.Figure()
go_d.plot_pd_deterministic(fig, det, production_datetimes=production_datetimes, colorscale=px.colors.sequential.Jet, color_loops=12, showlegend=False)
go_d.add_observed_trace(fig, obs, showlegend=False)
go_d.apply_default_layout(fig, yaxis_title="Q [m3/s]")

## Create ensemble forecasts
Based on probabilistic forecasts sampled with autoregressive p-value sampling.  

Requires some functions for efficiency. These are defined below.

In [ ]:
def generate_p_values(df, p_nan=0.9, n_min=2, **kwargs):
    p_values = df.copy()
    p_values.loc[:, :] = np.random.uniform(0, 1, size=p_values.shape)

    mask = np.random.binomial(n=1, p=1-p_nan, size=p_values.shape)
    mask[:, [0, -1]] = 1

    while (mask.sum(axis=1)<n_min).any():
        idxs = np.where(mask.sum(axis=1)<n_min)[0]
        mask[idxs, 1:-2] = np.random.binomial(n=1, p=1-p_nan, size=mask[idxs, 1:-2].shape)

    p_values = p_values.mask(~mask.astype(bool))
    p_values = p_values.interpolate(axis=1, **kwargs)
    
    return p_values

def interp_rows_matrix(x_values, xp, fp_rows):
    '''
    Vectorized row-wise equivalent of:

        np.interp(x_values[row], xp, fp_rows[row],
                  left=fp_rows[row, 0],
                  right=fp_rows[row, -1])

    Parameters
    ----------
    x_values : array, shape (n_rows, n_targets)
        Values to interpolate at. In your case: p_values_n[ens].values

    xp : array, shape (n_bands,)
        Common x-grid. In your case: probability_bands

    fp_rows : array, shape (n_rows, n_bands)
        Row-specific y-values. In your case: ensemble_data.values

    Returns
    -------
    out : array, shape (n_rows, n_targets)
    '''
    
    x_values = np.asarray(x_values, dtype=float)
    xp = np.asarray(xp, dtype=float)
    fp_rows = np.asarray(fp_rows, dtype=float)

    n_rows, n_targets = x_values.shape
    row_idx = np.arange(n_rows)[:, None]

    # Find interval indices
    j = np.searchsorted(xp, x_values, side="right")

    # Handle left/right extrapolation like np.interp
    left_mask = j == 0
    right_mask = j == len(xp)

    # Clip to valid interpolation interval
    j = np.clip(j, 1, len(xp) - 1)

    j0 = j - 1
    j1 = j

    x0 = xp[j0]
    x1 = xp[j1]

    y0 = fp_rows[row_idx, j0]
    y1 = fp_rows[row_idx, j1]

    w = (x_values - x0) / (x1 - x0)

    out = y0 * (1 - w) + y1 * w

    # Match np.interp left/right behavior
    out[left_mask] = fp_rows[row_idx, 0][left_mask]
    out[right_mask] = fp_rows[row_idx, -1][right_mask]

    return out

In [ ]:
ens_path = test_dataset_path / 'ens.parquet'
if ens_path.exists():
    ens = pd.read_parquet(ens_path)
else:
    n_ensembles = 10
    p_nan=0.95
    # interpolate_kwargs = dict(
    #     method='quadratic',
    #     n_min=3,
    # )
    interpolate_kwargs = dict(
        method='linear',
    )

    probability_bands = prob.index.get_level_values('non_exceedance').unique()

    ensemble_data = prob.unstack('non_exceedance')
    # display(ensemble_data)

    p_values_shape = prob.unstack('non_exceedance').iloc[:, 0].droplevel('event_datetime')
    p_value_mask = p_values_shape.unstack('leadtime')
    p_value_mask.loc[:, :] = 1
    p_values_n = [(p_value_mask * generate_p_values(p_value_mask, p_nan=p_nan, **interpolate_kwargs)).stack('leadtime').to_frame(name='p-value') for i in range(n_ensembles)]

    idx = pd.MultiIndex.from_frame(ensemble_data.index.to_frame(index=False).loc[:, ['production_datetime', 'leadtime']])
    # display(idx)
    for i0 in range(len(p_values_n)):
        p_values_n[i0] = p_values_n[i0].reindex(idx) 
    # display(p_values_n[0])

    ensembles = []
    for i0, pv in enumerate(p_values_n):
        tmp = pd.DataFrame(
            interp_rows_matrix(pv.to_numpy(dtype=float), np.asarray(probability_bands, dtype=float), ensemble_data.to_numpy(dtype=float)),
            index=pv.index,
            columns=pd.Index([i0], name='ensemble_member')
        )
        ensembles.append(tmp)

    ensembles = pd.concat(ensembles, axis=1)
    ensembles.index = ensemble_data.index
    ens = ensembles.stack('ensemble_member').to_frame(name='Q')

    ens.to_parquet(ens_path)

display(ens)

In [ ]:
production_datetimes = ens.index.get_level_values("production_datetime").unique()[::24]

fig = go.Figure()
go_d.plot_pd_ensemble(fig, ens, production_datetimes=production_datetimes, colors=px.colors.qualitative.G10, color_loops=10)
go_d.add_observed_trace(fig, obs, showlegend=False)
go_d.apply_default_layout(fig, yaxis_title="Q [m3/s]")
fig.show()